# Cross-Framework Comparison - Logistic Regression

In [1]:
# Set path of the custom jar that contains the new models
import os
import sys

# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [2]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Global imports

In [3]:
from river import linear_model, optim
from tabulate import tabulate
import importlib.util
import sys
import os
import time
import tracemalloc

# Dynamically load the custom LogisticRegression over the installed capymoa package
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

from capymoa.evaluation import prequential_evaluation

# We use an external library (scikit-learn) to compute the metrics consistently across models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

Global functions

In [4]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [5]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Accuracy", "F1", "Precision", "Recall"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    for key in ["Time (s)", "Memory (MB)"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.4f}",
            f"{river:.4f}",
            f"{(capy - river):+.4f}"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using CapyMOA built-in function
    results = prequential_evaluation(
        stream=stream,
        learner=log_reg_capymoa,
        max_instances=MAX_INSTANCES
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = {
        "Accuracy": results['cumulative'].accuracy(),
        "F1": results['cumulative'].f1_score(),
        "Precision": results['cumulative'].precision(),
        "Recall": results['cumulative'].recall(),
        "Time (s)": end_time - start_time,
        "Memory (MB)": peak_memory / (1024 * 1024)
    }

    return metrics

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    y_true = []
    y_pred = []

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation
    for i, (x, y) in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        pred = log_reg_river.predict_one(x)

        if pred is None:
            pred = False

        y_true.append(y)
        y_pred.append(pred)

        log_reg_river.learn_one(x, y)

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = _compute_sklearn_metrics(y_true, y_pred)
    metrics["Time (s)"] = end_time - start_time
    metrics["Memory (MB)"] = peak_memory / (1024 * 1024)

    return metrics

def _compute_sklearn_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "F1": f1_score(y_true, y_pred, zero_division=0) * 100,
        "Precision": precision_score(y_true, y_pred, zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, zero_division=0) * 100,
    }

## Electricity dataset

In [6]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 69.93%    │ 69.93%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 68.83%    │ 75.02%  │ -6.19%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 69.26%    │ 71.86%  │ -2.59%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 68.41%    │ 78.48%  │ -10.07% │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 1.7052    │ 6.7478  │ -5.0426 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0448    │ 0.7573  │ -0.7124 │
╘═════════════╧═══════════╧═════════╧═════════╛


## ElectricityTiny dataset

In [7]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 62.85%    │ 62.85%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 58.67%    │ 38.85%  │ +19.82% │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 60.27%    │ 55.79%  │ +4.48%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 57.16%    │ 29.80%  │ +27.36% │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.0937    │ 0.2038  │ -0.1102 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0023    │ 0.0332  │ -0.0309 │
╘═════════════╧═══════════╧═════════╧═════════╛


## RandomRBFGenerator

In [8]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Accuracy    │ 85.28%    │ 85.28%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 84.75%    │ 81.59%  │ +3.16%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Precision   │ 85.19%    │ 84.83%  │ +0.36%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Recall      │ 84.31%    │ 78.59%  │ +5.72%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Time (s)    │ 0.5091    │ 24.1377 │ -23.6286 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.0118    │ 1.5382  │ -1.5263  │
╘═════════════╧═══════════╧═════════╧══════════╛


## Hyper100k dataset

In [9]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Accuracy    │ 89.86%    │ 89.86%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 89.92%    │ 90.15%  │ -0.23%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Precision   │ 89.98%    │ 87.85%  │ +2.13%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Recall      │ 89.85%    │ 92.57%  │ -2.71%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Time (s)    │ 1.0074    │ 11.6950 │ -10.6876 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.0118    │ 1.5316  │ -1.5198  │
╘═════════════╧═══════════╧═════════╧══════════╛


## SEA dataset generator

In [10]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 82.91%    │ 82.91%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 81.09%    │ 87.05%  │ -5.96%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 81.92%    │ 84.69%  │ -2.77%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 80.28%    │ 89.56%  │ -9.28%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.2386    │ 8.7842  │ -8.5456 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0124    │ 1.5401  │ -1.5277 │
╘═════════════╧═══════════╧═════════╧═════════╛


## HyperPlaneClassification dataset

In [11]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Accuracy    │ 89.87%    │ 89.87%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 89.94%    │ 90.18%  │ -0.24%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Precision   │ 90.01%    │ 87.71%  │ +2.30%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Recall      │ 89.86%    │ 92.78%  │ -2.92%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Time (s)    │ 0.0985    │ 10.9130 │ -10.8145 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.0118    │ 1.5313  │ -1.5195  │
╘═════════════╧═══════════╧═════════╧══════════╛


## RandomTreeGenerator

In [12]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Accuracy    │ 81.05%    │ 81.05%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 77.21%    │ 66.09%  │ +11.12%  │
├─────────────┼───────────┼─────────┼──────────┤
│ Precision   │ 79.66%    │ 76.99%  │ +2.67%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Recall      │ 74.89%    │ 57.89%  │ +17.01%  │
├─────────────┼───────────┼─────────┼──────────┤
│ Time (s)    │ 0.0811    │ 10.2824 │ -10.2013 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.0117    │ 1.5402  │ -1.5285  │
╘═════════════╧═══════════╧═════════╧══════════╛


## Electricity dataset (changed model parameters)

### L2

In [13]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l2=0.01)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 68.49%    │ 68.49%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 67.21%    │ 74.33%  │ -7.12%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 67.85%    │ 69.97%  │ -2.12%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 66.58%    │ 79.27%  │ -12.69% │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.2094    │ 6.1914  │ -5.9821 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0068    │ 0.7563  │ -0.7495 │
╘═════════════╧═══════════╧═════════╧═════════╛


### L1

In [14]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l1=0.01)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 65.76%    │ 65.76%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 64.06%    │ 73.34%  │ -9.28%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 65.27%    │ 66.43%  │ -1.16%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 62.89%    │ 81.86%  │ -18.97% │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.2220    │ 7.2747  │ -7.0527 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0073    │ 0.7579  │ -0.7506 │
╘═════════════╧═══════════╧═════════╧═════════╛


### Learning rate

In [15]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 81.08%    │ 81.08%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 80.57%    │ 83.76%  │ -3.19%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 80.71%    │ 82.78%  │ -2.06%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 80.43%    │ 84.76%  │ -4.33%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.2810    │ 4.6787  │ -4.3978 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0067    │ 0.7559  │ -0.7492 │
╘═════════════╧═══════════╧═════════╧═════════╛


### Gradient clipping

In [16]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, clip=1)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Accuracy    │ 69.93%    │ 69.93%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 68.83%    │ 75.02%  │ -6.19%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Precision   │ 69.26%    │ 71.86%  │ -2.59%  │
├─────────────┼───────────┼─────────┼─────────┤
│ Recall      │ 68.41%    │ 78.48%  │ -10.07% │
├─────────────┼───────────┼─────────┼─────────┤
│ Time (s)    │ 0.2518    │ 4.6126  │ -4.3608 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0073    │ 0.7558  │ -0.7485 │
╘═════════════╧═══════════╧═════════╧═════════╛


### Bias initialization

In [17]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, bias_init=3)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Accuracy    │ 70.62%    │ 70.62%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 69.55%    │ 75.65%  │ -6.10%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Precision   │ 70.02%    │ 72.33%  │ -2.31%   │
├─────────────┼───────────┼─────────┼──────────┤
│ Recall      │ 69.09%    │ 79.28%  │ -10.19%  │
├─────────────┼───────────┼─────────┼──────────┤
│ Time (s)    │ 0.2169    │ 10.8751 │ -10.6581 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.0072    │ 0.7571  │ -0.7499  │
╘═════════════╧═══════════╧═════════╧══════════╛
